### 01: Attention Manual Math

In [1]:
import torch 
import torch.nn.functional as F

X = torch.tensor([
    [1., 0., 1.],
    [0., 2., 1.],
    [1., 1., 0.]
])

scores  = X @ X.T
weights = F.softmax(scores, dim=-1)
out     = weights @ X

print("Scores\n", scores)
print("Weights\n", weights)
print("Output\n", out)

Scores
 tensor([[2., 1., 1.],
        [1., 5., 2.],
        [1., 2., 2.]])
Weights
 tensor([[0.5761, 0.2119, 0.2119],
        [0.0171, 0.9362, 0.0466],
        [0.1554, 0.4223, 0.4223]])
Output
 tensor([[0.7881, 0.6358, 0.7881],
        [0.0638, 1.9191, 0.9534],
        [0.5777, 1.2670, 0.5777]])


### 02: Attention Numpy

In [2]:
import numpy as np 

def softmax(x):
    e = np.exp(x - x.max(axis=-1, keepdims=True))
    return e / e.sum(axis=-1, keepdims=True)

Q = np.random.randn(4, 8)
K = np.random.randn(4, 8)
V = np.random.randn(4, 8)

W = softmax(Q @ K.T / np.sqrt(Q.shape[-1]))

print((W @ V).shape)

(4, 8)


### 03: QKV Projection

In [3]:
import torch
import torch.nn as nn

x = torch.randn(2, 5, 32)
q_proj, k_proj, v_proj = nn.Linear(32, 32), nn.Linear(32, 32), nn.Linear(32, 32)
Q, K, V = q_proj(x), k_proj(x), v_proj(x)

print(Q.shape, K.shape, V.shape)

torch.Size([2, 5, 32]) torch.Size([2, 5, 32]) torch.Size([2, 5, 32])


### 04: Scaled Dot Product Attention

In [4]:
import torch, math
import torch.nn.functional as F

def attention(Q, K, V, mask=None):
    scores = Q @ K.transpose(-2, -1) / math.sqrt(Q.size(-1))
    if mask is not None:
        scores = scores.masked_fill(mask == 0, float('-inf'))
    weights = F.softmax(scores, dim=-1)
    return weights @ V, weights

Q = torch.randn(2, 4, 8)
K = torch.randn(2, 4, 8)
V = torch.randn(2, 4, 8)

output, weights = attention(Q, K, V)

print(output.shape)
print(weights.shape) 

torch.Size([2, 4, 8])
torch.Size([2, 4, 4])


### 05: Self Attention

In [5]:
import torch, math
import torch.nn as nn

class SelfAttention(nn.Module):
    def __init__(self, d_model):
        super().__init__()
        self.q = nn.Linear(d_model, d_model)
        self.k = nn.Linear(d_model, d_model)
        self.v = nn.Linear(d_model, d_model)
        self.d_model = d_model
    def forward(self, x):
        Q, K, V = self.q(x), self.k(x), self.v(x)
        w = (Q @ K.transpose(-2, -1) / math.sqrt(self.d_model)).softmax(-1)
        return w @ V, w



In [7]:
batch_size = 2
seq_len    = 5
d_model    = 8

x = torch.randn(batch_size, seq_len, d_model)

attn = SelfAttention(d_model)

output, weights = attn(x)

print("Input   Shape : ", x.shape)
print("Output  Shape : ", output.shape)
print("Weights Shape : ", weights.shape)

Input   Shape :  torch.Size([2, 5, 8])
Output  Shape :  torch.Size([2, 5, 8])
Weights Shape :  torch.Size([2, 5, 5])


### 06: Multi-Head Attention

In [15]:
import torch, math
import torch.nn as nn

class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, n_heads, causal=False):
        super().__init__()
        assert d_model % n_heads == 0, "d_model must be divisible by n_heads"
        self.h = n_heads; self.d = d_model//n_heads; self.causal = causal
        self.qkv = nn.Linear(d_model, 3 * d_model)
        self.out = nn.Linear(d_model, d_model)

    def forward(self, x, context = None):
        if context is not None: raise NotImplementedError("Use context for cross attention")
        B, T, C = x.shape
        Q, K, V = self.qkv(x).chunk(3, -1)
        def split(z): 
            return z.view(B, T, self.h, self.d).transpose(1, 2)
        Q,K,V  = map(split,(Q,K,V))
        scores = Q @ K.transpose(-2,-1)/math.sqrt(self.d)
        if self.causal:
            mask   = torch.tril(torch.ones(T,T,device=x.device,dtype=torch.bool))
            scores = scores.masked_fill(~mask,float("-inf"))
        w = scores.softmax(-1)
        y = (w@V).transpose(1,2).contiguous().view(B,T,C)
        return self.out(y)

batch_size = 2
seq_len    = 5
d_model    = 8
n_heads    = 2

x = torch.randn(batch_size, seq_len, d_model)
mha = MultiHeadAttention(d_model, n_heads, causal=True)
output = mha(x)

print(output.shape)

torch.Size([2, 5, 8])


### 07: MHA einsum

In [ ]:
import torch, math

# B = Batch Size
# H = Number of Heads
# T = Sequence Length
# D = Model Dimension
B, H, T, D = 2, 4, 5, 8
Q, K, V    = [torch.randn(B, H, T, D) for _ in range(3)]
scores     = torch.einsum("bhtd, bhsd -> bhts", Q, K) / math.sqrt(D)
weights    = scores.softmax(-1)
out        = torch.einsum("bhts, bhsd -> bhtd", weights, V)
print(out.shape)


torch.Size([2, 4, 5, 8])


### 08: Additive Attention

In [19]:
import torch
import torch.nn as nn

class AdditiveAttention(nn.Module):
    def __init__(self, d):
        super().__init__(); self.Wq = nn.Linear(d, d); self.Wk = nn.Linear(d, d); self.v = nn.Linear(d, 1)
    def forward(self, Q, K, V):
        scores = self.v(
            torch.tanh(self.Wq(Q).unsqueeze(2) + self.Wk(K).unsqueeze(1))
        ).squeeze(-1)
        w      = scores.softmax(-1)
        return w @ V

batch_size = 2
Tq = 4 # query sequence length
Tk = 6 # key/value length
d  = 8 # embedding dimension

Q  = torch.randn(batch_size, Tq, d)
K  = torch.randn(batch_size, Tk, d)
V  = torch.randn(batch_size, Tk, d)

attn = AdditiveAttention(d)
output = attn(Q, K, V)

print(output.shape)

torch.Size([2, 4, 8])


### 09 Cross Attention

In [22]:
import torch, math
import torch.nn as nn

class CrossAttention(nn.Module):
    def __init__(self, d_model):    
        super().__init__()
        self.q = nn.Linear(d_model, d_model); self.k = nn.Linear(d_model, d_model); self.v = nn.Linear(d_model, d_model)

    def forward(self, query_x, context_x):
        Q, K, V = self.q(query_x), self.k(context_x), self.v(context_x)
        scores  = Q @ K.transpose(-2, -1) / math.sqrt(Q.size(-1))
        weights = scores.softmax(-1)
        return weights @ V, weights

batch_size = 2
Tq = 4
Tk = 6 
d_model = 8

query_x   = torch.randn(batch_size, Tq, d_model)
context_x = torch.randn(batch_size, Tk, d_model)

attn   = CrossAttention(d_model)
output, weights = attn(query_x, context_x)

print(f"Output  shape: {output.shape}")
print(f"Weights shape: {weights.shape}")

Output  shape: torch.Size([2, 4, 8])
Weights shape: torch.Size([2, 4, 6])


### 10 Casual Attention

In [23]:
import torch
def casual_mask(T, device = None):
    return torch.tril(torch.ones(T, T, dtype=torch.bool, device=device))

mask = casual_mask(5)
print(mask)

tensor([[ True, False, False, False, False],
        [ True,  True, False, False, False],
        [ True,  True,  True, False, False],
        [ True,  True,  True,  True, False],
        [ True,  True,  True,  True,  True]])


### 11 Padding Mask

In [25]:
import torch

tokens = torch.tensor([
    [5, 8, 2, 0, 0],
    [1, 4, 7, 9, 0]
])

mask = tokens != 0
print(mask)

tensor([[ True,  True,  True, False, False],
        [ True,  True,  True,  True, False]])


### 12 Local Attention

In [26]:
import torch 
def local_mask(T, window_size, device=None):
    mask = torch.zeros(T, T, dtype=torch.bool, device=device)
    for i in range(T):
        start = max(0, i - window_size)
        end = min(T, i + window_size + 1)
        mask[i, start:end] = 1
    return mask
print(local_mask(8, 2).int())

tensor([[1, 1, 1, 0, 0, 0, 0, 0],
        [1, 1, 1, 1, 0, 0, 0, 0],
        [1, 1, 1, 1, 1, 0, 0, 0],
        [0, 1, 1, 1, 1, 1, 0, 0],
        [0, 0, 1, 1, 1, 1, 1, 0],
        [0, 0, 0, 1, 1, 1, 1, 1],
        [0, 0, 0, 0, 1, 1, 1, 1],
        [0, 0, 0, 0, 0, 1, 1, 1]], dtype=torch.int32)


### 13 Multi Query Attention

In [27]:
import torch

B, H, T, D = 2, 8, 5, 16

Q = torch.randn(B, H, T, D)
K = V = torch.randn(B, 1, T, D)
scores = ( Q @ K.transpose(-2, -1)) / D ** 0.5

print((scores.softmax(-1)@V).shape)

torch.Size([2, 8, 5, 16])


### 14 Grouped Query Attention

In [35]:
import torch, math
import torch.nn as nn

class GroupedQueryAttention(nn.Module):
    def __init__(self, d_model, n_q_heads, n_kv_heads):
        super().__init__()
        assert d_model % n_q_heads == 0, "d_model must be divisible by n_q_heads"
        assert n_q_heads % n_kv_heads == 0, "n_q_heads must be a multiple of n_kv_heads"

        self.d_model    = d_model
        self.n_q_heads  = n_q_heads
        self.n_kv_heads = n_kv_heads
        self.d_head     = d_model // n_q_heads   # per‑head dimension (shared by Q and K/V)

        # Linear projections
        self.q_proj   = nn.Linear(d_model, d_model)
        self.k_proj   = nn.Linear(d_model, n_kv_heads * self.d_head)
        self.v_proj   = nn.Linear(d_model, n_kv_heads * self.d_head)
        self.out_proj = nn.Linear(d_model, d_model)

    def forward(self, x):
        B, T, C = x.shape

        # Project Q, K, V
        Q = self.q_proj(x).view(B, T, self.n_q_heads, self.d_head).transpose(1, 2)   # (B, n_q_heads, T, d_head)
        K = self.k_proj(x).view(B, T, self.n_kv_heads, self.d_head).transpose(1, 2)  # (B, n_kv_heads, T, d_head)
        V = self.v_proj(x).view(B, T, self.n_kv_heads, self.d_head).transpose(1, 2)  # (B, n_kv_heads, T, d_head)

        # Map query heads to kv groups
        group_size = self.n_q_heads // self.n_kv_heads
        K = K.repeat_interleave(group_size, dim=1)  # (B, n_q_heads, T, d_head)
        V = V.repeat_interleave(group_size, dim=1)  # (B, n_q_heads, T, d_head)

        # Attention scores
        scores = (Q @ K.transpose(-2, -1)) / math.sqrt(self.d_head)  # (B, n_q_heads, T, T)
        weights = scores.softmax(-1)

        # Weighted sum
        out = (weights @ V).transpose(1, 2).contiguous().view(B, T, C)
        return self.out_proj(out)


batch_size = 2
seq_len    = 5
d_model    = 16
n_q_heads  = 8
n_kv_heads = 2

x = torch.randn(batch_size, seq_len, d_model)
gpa = GroupedQueryAttention(d_model, n_q_heads, n_kv_heads)
output = gpa(x)

print(output.shape)

torch.Size([2, 5, 16])


### 15 Attention Debugger

In [30]:
import torch, math
import torch.nn.functional as F

def attention(Q, K, V, mask=None):
    # Q, K, V shapes: (batch, seq_len, d_model)
    scores = Q @ K.transpose(-2, -1) / math.sqrt(Q.size(-1))  # (batch, seq_len, seq_len)
    if mask is not None:
        scores = scores.masked_fill(mask == 0, float('-inf'))
    weights = F.softmax(scores, dim=-1)                       # attention weights
    out = weights @ V                                         # weighted sum of values
    return out, weights

Q = K = V = torch.randn(2, 4, 8)   
out, w = attention(Q, K, V)

print("Q shape:", Q.shape)         
print("K shape:", K.shape)         
print("V shape:", V.shape)         
print("Output shape:", out.shape)  
print("Weights shape:", w.shape)   
print("Weights sum to 1:", w.sum(-1))  


Q shape: torch.Size([2, 4, 8])
K shape: torch.Size([2, 4, 8])
V shape: torch.Size([2, 4, 8])
Output shape: torch.Size([2, 4, 8])
Weights shape: torch.Size([2, 4, 4])
Weights sum to 1: tensor([[1.0000, 1.0000, 1.0000, 1.0000],
        [1.0000, 1.0000, 1.0000, 1.0000]])
